# CS-GY 6643 — Project 3: Questions 1–3

Jim Lo and Ta-Chun Tai

We compare CNN representations of the tiger and portrait images, estimate vehicle motion in the highway video, and track the person and dog through the second video. Each question includes the method, results, and our interpretation.

To reproduce the results, select a Colab T4 GPU and run the notebook from top to bottom. Both video pipelines process every frame. Figures, per-frame measurements, and annotated videos are saved in `output`; the last results section writes a ZIP archive.

## Setup

We use ImageNet-pretrained ResNet50 and DenseNet121 for Question 1, COCO-pretrained YOLO11m-seg for video detection, and SAM2.1 base for temporal segmentation in Question 3. All models use pretrained weights without fine-tuning.

The cells below install the required packages, read the supplied inputs and check their sizes and video metadata. They also record the software versions, input hashes, and random seed for reproducibility.

In [ ]:
%pip -q install 'ultralytics==8.3.221'

In [ ]:
from pathlib import Path
import os, sys, json, time, random, hashlib, platform, subprocess, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Video, HTML
import torch
import torchvision
import torchvision.transforms.functional as TF
import cv2

SEED = 6643
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
ROOT = Path.cwd()
DATA, OUT = ROOT / 'data', ROOT / 'output'
for directory in [DATA, OUT, OUT/'q1', OUT/'q2', OUT/'q3']:
    directory.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 160, 'axes.grid': False})
cv2.setNumThreads(2)

ASSETS = {
    'Royal-Bengal-Tiger.jpg': 114261,
    'face.png': 189462,
    'highway.mp4': 35662657,
    'video.mp4': 4732432,
}
manifest = []
for name, expected_size in ASSETS.items():
    path = DATA / name
    assert path.is_file() and path.stat().st_size == expected_size, f'Incomplete asset download: {name}'
    manifest.append({'file': name, 'bytes': path.stat().st_size,
                     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
pd.DataFrame(manifest).to_csv(OUT/'input_manifest.csv', index=False)

def video_info(path):
    cap = cv2.VideoCapture(str(path))
    assert cap.isOpened(), f'Cannot open video: {path}'
    info = dict(frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                fps=float(cap.get(cv2.CAP_PROP_FPS)),
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    info['duration_s'] = info['frames']/info['fps']
    return info

video_metadata = {name: video_info(DATA/name) for name in ['highway.mp4', 'video.mp4']}
assert video_metadata['highway.mp4']['frames'] == 1501
assert abs(video_metadata['highway.mp4']['fps']-25) < 1e-6
versions = {p: importlib.metadata.version(p) for p in
            ['torch','torchvision','ultralytics','opencv-python','numpy','pandas']}
environment = {'python': sys.version, 'device': DEVICE, 'versions': versions,
               'gpu': torch.cuda.get_device_name(0) if DEVICE.startswith('cuda') else None,
               'seed': SEED, 'videos': video_metadata}
(OUT/'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
display(Markdown(f'**Device:** `{DEVICE}` | GPU: `{environment["gpu"]}`. Original input metadata is shown below.'))
display(pd.DataFrame(video_metadata).T)
display(pd.DataFrame(manifest)[['file','bytes','sha256']])
print(json.dumps(versions, indent=2))

def show_saved(path, width=1000):
    from IPython.display import Image as IPImage
    display(IPImage(filename=str(path), width=width))

def read_frame(path, index):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = cap.read()
    cap.release()
    assert ok, (str(path), index)
    return frame

## 1. CNN Feature Visualization and Interpretation

### 1(a)–(b) Feature maps and Grad-CAM

We give both networks the same input: resize the image to a short side of 256 pixels, take a 224 × 224 center crop, and apply ImageNet normalization. The displayed crops show exactly what each model sees.

For ResNet50, we inspect `layer1`, `layer3`, and `layer4`; for DenseNet121, we inspect `features.denseblock1`, `features.denseblock3`, and `features.denseblock4`. Each 4 × 4 grid shows channels 0–15, including inactive channels. Positive responses are scaled separately within each channel, so brightness shows where a channel responds rather than its strength relative to other channels. Sparsity statistics use all channels.

Grad-CAM explains the top predicted ImageNet class. We average the gradients spatially, use them to weight the final feature maps, sum the weighted maps, and apply ReLU. Alongside the combined heatmap, we show 16 individual channel contributions. For the portrait, the target class must be considered when interpreting the heatmap: these classifiers were not trained for face recognition.

In [ ]:
from torchvision.models import resnet50, densenet121, ResNet50_Weights, DenseNet121_Weights
from torchvision.transforms import InterpolationMode

# Use the same 224x224 crop for both models to control preprocessing differences.
inputs_q1 = {}
for image_name, file_name in [('tiger','Royal-Bengal-Tiger.jpg'), ('face','face.png')]:
    original = Image.open(DATA/file_name).convert('RGB')
    crop = TF.center_crop(TF.resize(original, 256, interpolation=InterpolationMode.BILINEAR), [224,224])
    tensor = TF.normalize(TF.to_tensor(crop), [0.485,0.456,0.406], [0.229,0.224,0.225]).unsqueeze(0)
    inputs_q1[image_name] = (crop, tensor)
fig, axes = plt.subplots(1,2,figsize=(8,4))
for ax,(name,(crop,_)) in zip(axes,inputs_q1.items()):
    ax.imshow(crop); ax.set_title(f'{name}: actual model input'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT/'q1'/'model_inputs.png'); plt.show(); plt.close(fig)

feature_rows, prediction_rows, cam_rows = [], [], []
q1_cam_data = {}
specs = [
    ('ResNet50', resnet50, ResNet50_Weights.IMAGENET1K_V1,
     ['layer1','layer3','layer4']),
    ('DenseNet121', densenet121, DenseNet121_Weights.IMAGENET1K_V1,
     ['features.denseblock1','features.denseblock3','features.denseblock4']),
]
for model_name, constructor, weights, layer_names in specs:
    model = constructor(weights=weights).eval().to(DEVICE)
    modules = dict(model.named_modules())
    for image_name,(crop,input_tensor) in inputs_q1.items():
        activations, gradients, handles = {}, {}, []
        def make_hook(layer_name):
            def hook(module, args, output):
                activations[layer_name] = output.detach().clone()
                if layer_name == layer_names[-1]:
                    output.register_hook(lambda grad: gradients.update({layer_name: grad.detach().clone()}))
            return hook
        for layer_name in layer_names:
            handles.append(modules[layer_name].register_forward_hook(make_hook(layer_name)))
        try:
            model.zero_grad(set_to_none=True)
            logits = model(input_tensor.to(DEVICE))
            probabilities = logits.softmax(dim=1)
            values, indices = probabilities[0].topk(5)
            target_class = int(indices[0])
            target_name = weights.meta['categories'][target_class]
            prediction_rows.append({'model':model_name,'image':image_name,'target_class':target_name,
                                    'probability':float(values[0]),
                                    'top5':'; '.join(f'{weights.meta["categories"][int(c)]}: {float(p):.4f}'
                                                   for p,c in zip(values,indices))})
            logits[0,target_class].backward()
            for depth,layer_name in zip(['early','mid','deep'],layer_names):
                raw = activations[layer_name][0].float().cpu().numpy()
                # DenseNet block outputs are signed; use positive responses for comparable statistics.
                positive = np.maximum(raw,0)
                assert np.isfinite(raw).all()
                maximum = positive.max(axis=(1,2), keepdims=True)
                normalized = np.divide(positive, maximum, out=np.zeros_like(positive), where=maximum>0)
                feature_rows.append({'model':model_name,'image':image_name,'depth':depth,'layer':layer_name,
                                     'channels':raw.shape[0],'height':raw.shape[1],'width':raw.shape[2],
                                     'nonpositive_fraction':float((raw<=0).mean()),
                                     'relative_inactive_fraction':float((normalized<=0.05).mean()),
                                     'dead_channel_fraction':float((maximum[:,0,0]<=1e-12).mean())})
                # Show the first 16 fixed channels, retaining inactive channels rather than cherry-picking.
                fig,axes=plt.subplots(4,4,figsize=(8,8))
                for ch,ax in enumerate(axes.flat):
                    ax.imshow(normalized[ch],cmap='viridis',vmin=0,vmax=1,interpolation='nearest')
                    ax.set_title(f'ch {ch}',fontsize=9);ax.axis('off')
                fig.suptitle(f'{model_name} | {image_name} | {layer_name}\n'
                             f'positive response, 16 fixed channels, spatial size {raw.shape[1:]}',fontsize=12)
                fig.tight_layout(rect=[0,0,1,0.93])
                fig.savefig(OUT/'q1'/f'{model_name}_{image_name}_{depth}_16channels.png')
                plt.show();plt.close(fig)
            A = activations[layer_names[-1]][0].float()
            G = gradients[layer_names[-1]][0].float()
            alpha = G.mean(dim=(1,2))
            cam_native = torch.relu((alpha[:,None,None]*A).sum(dim=0)).cpu().numpy()
            assert np.isfinite(cam_native).all() and cam_native.max()>0, 'Degenerate empty Grad-CAM'
            cam_native = cam_native / (cam_native.max()+1e-12)
            cam = cv2.resize(cam_native,(224,224),interpolation=cv2.INTER_LINEAR)
            rgb = np.asarray(crop)/255.
            heat = plt.get_cmap('jet')(cam)[...,:3]
            overlay = np.clip(0.6*rgb+0.4*heat,0,1)
            fig,axes=plt.subplots(1,3,figsize=(10,3.6))
            for ax,im,title in zip(axes,[rgb,cam_native,overlay],['Model input','Native Grad-CAM (7x7)','Grad-CAM overlay']):
                ax.imshow(im,cmap='jet' if im.ndim==2 else None,interpolation='nearest');ax.set_title(title);ax.axis('off')
            fig.suptitle(f'{model_name} | {image_name} | target: {target_name} ({float(values[0]):.1%})',fontsize=11)
            fig.tight_layout(rect=[0,0,1,0.90]);fig.savefig(OUT/'q1'/f'{model_name}_{image_name}_gradcam.png')
            plt.show();plt.close(fig)
            # Match the provided per-channel example while also reporting standard aggregate Grad-CAM.
            top_channels = alpha.topk(16).indices.cpu().tolist()
            fig,axes=plt.subplots(4,4,figsize=(9,9))
            for ch,ax in zip(top_channels,axes.flat):
                part = torch.relu(alpha[ch]*A[ch]).cpu().numpy()
                part = part/(part.max()+1e-12)
                part=cv2.resize(part,(224,224))
                ax.imshow(np.clip(.6*rgb+.4*plt.get_cmap('jet')(part)[...,:3],0,1))
                ax.set_title(f'ch {ch}, weight {float(alpha[ch]):.2g}',fontsize=8);ax.axis('off')
            fig.suptitle(f'{model_name} | {image_name} | deep per-channel Grad-CAM (top 16 weights)',fontsize=11)
            fig.tight_layout(rect=[0,0,1,.95]);fig.savefig(OUT/'q1'/f'{model_name}_{image_name}_channel_cams.png')
            plt.show();plt.close(fig)
            q1_cam_data[(model_name,image_name)]=cam
            cam_rows.append({'model':model_name,'image':image_name,'target':target_name,
                             'peak_x':int(np.unravel_index(cam.argmax(),cam.shape)[1]),
                             'peak_y':int(np.unravel_index(cam.argmax(),cam.shape)[0]),
                             'area_above_half_max':float((cam>=.5).mean())})
        finally:
            for handle in handles: handle.remove()
    del model
    if torch.cuda.is_available():torch.cuda.empty_cache()
feature_stats=pd.DataFrame(feature_rows)
prediction_stats=pd.DataFrame(prediction_rows)
cam_stats=pd.DataFrame(cam_rows)
feature_stats.to_csv(OUT/'q1'/'feature_statistics.csv',index=False)
prediction_stats.to_csv(OUT/'q1'/'predictions.csv',index=False)
cam_stats.to_csv(OUT/'q1'/'cam_statistics.csv',index=False)
assert len(feature_stats)==12 and len(prediction_stats)==4
display(prediction_stats)
display(feature_stats)
display(cam_stats)
print('Q1_CHECK: 12 feature grids, 4 aggregate Grad-CAM figures, 4 per-channel Grad-CAM grids completed.')

### 1(c) Analysis and comparison

**i. Abstraction.** The early layers retain recognizable image structure. Several channels outline the tiger's stripes and body, while the portrait produces clear eye, nose, mouth, and face contours. Middle layers select combinations of textures and parts. The deepest maps respond over broader, coarser regions; Grad-CAM helps relate these responses to the predicted class. Matching channel numbers across the two networks do not represent matching features.

**ii. Sparsity.** We count a positive response as inactive when it is at or below 5% of its channel maximum. Fully inactive channels are included. The percentages below use every channel in each selected layer.

| Model | Depth | Tiger inactive (%) | Portrait inactive (%) |
|---|---|---:|---:|
| ResNet50 | Early | 38.6 | 27.9 |
| ResNet50 | Middle | 82.1 | 79.7 |
| ResNet50 | Deep | 50.9 | 55.6 |
| DenseNet121 | Early | 53.0 | 54.5 |
| DenseNet121 | Middle | 71.5 | 71.3 |
| DenseNet121 | Deep | 60.6 | 63.0 |

For both networks, the portrait has slightly sparser positive responses in the deepest selected block. The ordering changes earlier in ResNet50, where the tiger is sparser. Sparsity also peaks in the middle layers rather than increasing steadily with depth. Residual addition and feature concatenation can produce different response distributions, but two images are not enough to attribute the differences to architecture alone. DenseNet block outputs contain negative values, so the CSV also reports the raw nonpositive fraction.

**iii. Smoothness and resolution.** Both sets of selected layers have spatial sizes of 56 × 56, 14 × 14, and 7 × 7. Deeper layers combine information from larger receptive fields and lose fine detail. The nearest-neighbor feature plots expose this coarse grid. Upsampling Grad-CAM makes a smoother overlay without recovering the missing spatial detail.

**iv. Semantic focus.** Both models predict *tiger*: 80.4% for ResNet50 and 87.5% for DenseNet121. Their Grad-CAMs emphasize the head, neck, and upper body. ResNet50 spreads its response farther over the body and lower image, while DenseNet121 concentrates more on the head, neck, and shoulder. The area above half the heatmap maximum is 45.8% for ResNet50 and 24.4% for DenseNet121. This measures concentration, not localization accuracy.

For the portrait, both models predict *suit*, with probabilities of 26.2% and 34.7%. Their maps emphasize the lower face, neck, collar, and shoulder more than the eyes or hair. This makes sense for a clothing-related prediction. Grad-CAM indicates regions associated with that prediction; it is neither a segmentation mask nor proof that a region causes the decision.

**v. Image dependence.** The tiger gives the clearer deep-layer explanation because the predicted category matches the visible animal. The portrait is easy to recognize in early feature maps, but its final explanation concerns clothing. ImageNet-1K has no generic human-face category, and the portrait's top predictions are spread across clothing classes. Recognizable low-level structure therefore does not guarantee a face-centered final heatmap.

![Grad-CAM on the tiger image](../assets/gradcam.jpg)

## Motion estimation for Questions 2 and 3

We use dense Farneback flow between consecutive frames. Computing the field from the current frame back to the previous one lets us sample it directly with the current object's foreground mask.

For a foreground region $M_t$, mean displacement is

$$d_t = \frac{1}{|M_t|}\sum_{p\in M_t}\sqrt{u_t(p)^2+v_t(p)^2}.$$

We average the magnitude of each pixel's motion vector. Averaging the vectors first would allow opposite motions to cancel. If flow is computed on a smaller image, we restore horizontal and vertical vectors to the original video scale before measuring displacement.

Segmentation masks reduce background contamination. When segmentation is missing, a box-based region is used and recorded in the CSV. Frame 0 is unavailable because it has no preceding frame. The test below checks a known translation at two flow resolutions and verifies the conversion from px/frame to px/s.

In [ ]:
import collections, gzip, shutil
from tqdm.auto import tqdm

def dense_backward(current_gray, previous_gray, max_width):
    """Flow at current-frame pixels, pointing into the preceding frame.
    Return vectors in ORIGINAL input-video pixels per frame, even if resized.
    """
    h,w=current_gray.shape
    fw=min(w,max_width); fh=round(h*fw/w)
    current=cv2.resize(current_gray,(fw,fh),interpolation=cv2.INTER_AREA)
    previous=cv2.resize(previous_gray,(fw,fh),interpolation=cv2.INTER_AREA)
    flow=cv2.calcOpticalFlowFarneback(current,previous,None,.5,5,25,4,7,1.5,0)
    flow[...,0]*=w/fw
    flow[...,1]*=h/fh
    return flow

def polygon_mask(shape, box, polygon=None, original_size=None):
    h,w=shape
    ow,oh=original_size if original_size else (w,h)
    scale=np.array([w/ow,h/oh])
    mask=np.zeros((h,w),np.uint8)
    method='segmentation'
    if polygon is not None and len(polygon)>=3:
        points=np.rint(np.asarray(polygon)*scale).astype(np.int32)
        cv2.fillPoly(mask,[points],1)
    else:
        method='box_fallback'
        b=np.rint(np.asarray(box)*np.tile(scale,2)).astype(int)
        x1,y1=max(0,b[0]),max(0,b[1]);x2,y2=min(w,b[2]),min(h,b[3])
        mask[y1:y2,x1:x2]=1
    if mask.sum()>=50:
        eroded=cv2.erode(mask,np.ones((3,3),np.uint8))
        if eroded.sum()>=12:mask=eroded
    return mask.astype(bool),method

def mean_motion(flow, mask):
    if flow is None or mask is None or mask.sum()==0:return np.nan,0
    vectors=flow[mask]
    valid=np.isfinite(vectors).all(axis=1)
    if not valid.any():return np.nan,0
    # Mean of per-pixel vector magnitudes, not magnitude of the mean vector.
    return float(np.linalg.norm(vectors[valid],axis=1).mean()),int(valid.sum())

def motion_label(value,unit):
    return f'{value:.1f} {unit}' if np.isfinite(value) else f'N/A {unit}'

def track_color(track_id):
    rgb=np.array(plt.get_cmap('tab20')(int(track_id)%20)[:3])*255
    return tuple(int(v) for v in rgb[::-1])

def draw_label(frame, box, text, color, predicted=False, scale=.55, label_offset=(0,0)):
    h,w=frame.shape[:2]
    x1,y1,x2,y2=np.rint(box).astype(int)
    x1,x2=np.clip([x1,x2],0,w-1);y1,y2=np.clip([y1,y2],0,h-1)
    if predicted:
        for a,b in [((x1,y1),(x2,y1)),((x2,y1),(x2,y2)),((x2,y2),(x1,y2)),((x1,y2),(x1,y1))]:
            a,b=np.array(a,float),np.array(b,float)
            length=np.linalg.norm(b-a)
            for s in np.arange(0,length,8):
                p=a+(b-a)*s/max(length,1);q=a+(b-a)*min(s+4,length)/max(length,1)
                cv2.line(frame,tuple(p.astype(int)),tuple(q.astype(int)),color,1,cv2.LINE_AA)
    else:cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
    lines=text.split('\n');lineheight=round(24*scale/.55)
    widths=[cv2.getTextSize(t,cv2.FONT_HERSHEY_SIMPLEX,scale,1)[0][0] for t in lines]
    left=min(max(0,x1+label_offset[0]),max(0,w-max(widths)-8))
    top=max(0,y1+label_offset[1]-lineheight*len(lines)-3)
    cv2.rectangle(frame,(left,top),(min(w-1,left+max(widths)+8),top+lineheight*len(lines)),(20,20,20),-1)
    for j,line in enumerate(lines):
        cv2.putText(frame,line,(left+3,top+(j+1)*lineheight-5),cv2.FONT_HERSHEY_SIMPLEX,scale,color,1,cv2.LINE_AA)

def encode_video(raw_path, final_path):
    command=['ffmpeg','-y','-loglevel','error','-i',str(raw_path),'-an','-c:v','libx264',
             '-preset','fast','-crf','24','-pix_fmt','yuv420p','-movflags','+faststart',str(final_path)]
    completed=subprocess.run(command,capture_output=True,text=True)
    assert completed.returncode==0,completed.stderr[-2000:]
    assert final_path.exists() and final_path.stat().st_size>1000
    return video_info(final_path)

def display_contact(frames, path, columns=2, titles=None, size=(14,8)):
    fig,axes=plt.subplots((len(frames)+columns-1)//columns,columns,figsize=size,squeeze=False)
    for ax in axes.flat:ax.axis('off')
    for j,((idx,frame),ax) in enumerate(zip(frames.items(),axes.flat)):
        ax.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB));ax.set_title(titles[j] if titles else f'Frame {idx}')
    fig.tight_layout();fig.savefig(path);plt.show();plt.close(fig)

# A meaningful scale/unit check: recover a known synthetic translation.
test_rng=np.random.default_rng(SEED)
test_image=cv2.GaussianBlur(test_rng.integers(0,256,(240,320),dtype=np.uint8),(3,3),0)
dx,dy=6.,4.
test_next=cv2.warpAffine(test_image,np.float32([[1,0,dx],[0,1,dy]]),(320,240),borderMode=cv2.BORDER_REFLECT)
unit_rows=[]
for width in [320,160]:
    test_flow=dense_backward(test_next,test_image,width)
    test_mask=np.zeros(test_flow.shape[:2],bool)
    test_mask[20:-20,20:-20]=True
    measured,_=mean_motion(test_flow,test_mask)
    expected=float(np.hypot(dx,dy))
    unit_rows.append({'flow_width':width,'expected_px_frame':expected,'measured_px_frame':measured,
                      'measured_px_s_at_25fps':25*measured})
    assert abs(measured-expected)<.65,(width,measured,expected)
assert np.isclose(8.46*25,211.5)
display(pd.DataFrame(unit_rows))
print('FLOW_UNIT_CHECK passed: original pixel scale and px/frame-to-px/s conversion.')

   flow_width  expected_px_frame  measured_px_frame  measured_px_s_at_25fps
0         320           7.211103           7.210139              180.253482
1         160           7.211103           7.203169              180.079222

FLOW_UNIT_CHECK passed: original pixel scale and px/frame-to-px/s conversion.


## 2. Vehicle Detection, Tracking, and Speed Estimation

### 2(a)–(b) Detection and tracking

YOLO11m-seg processes every frame at inference size 1280, keeping cars, buses, and trucks. Class-agnostic nonmaximum suppression reduces duplicate boxes when the same vehicle receives different class labels. ByteTrack associates detections across frames, using weaker detections to recover existing tracks while requiring stronger evidence to start a new track. The output shows each vehicle's class, ID, box, and center trajectory.

### 2(c) Optical-flow speed

The highway video runs at 25 FPS. We measure foreground flow in the original 1920 × 1080 pixel coordinates and convert displacement to speed using $v_t=25d_t$ px/s. An exponential moving average with $\alpha=0.35$ makes the displayed labels easier to follow. The CSV retains the raw displacement, raw speed, and displayed value.

Box-center displacement is saved as a separate diagnostic; speed comes from dense optical flow. The playback video is encoded at 1280 × 720, while all measurements retain the original input-pixel scale.

In [ ]:
import yaml
from ultralytics import YOLO

Q2_CONFIG={'detector':'yolo11m-seg.pt','imgsz':1280,'confidence':.12,'nms_iou':.65,
           'flow_width':960,'fps':25.,'output_width':1280,'ema_alpha':.35}
tracker_config={'tracker_type':'bytetrack','track_high_thresh':.25,'track_low_thresh':.1,
                'new_track_thresh':.30,'track_buffer':60,'match_thresh':.80,'fuse_score':True}
tracker_path=ROOT/'q2_bytetrack.yaml'
tracker_path.write_text(yaml.safe_dump(tracker_config))
q2_detector=YOLO(Q2_CONFIG['detector'])
q2_rows,q2_frame_rows,q2_cache=[],[],[]
q2_history=collections.defaultdict(list)
q2_ema={}
q2_frames={}
q2_info=video_metadata['highway.mp4'];W,H=q2_info['width'],q2_info['height']
outW=Q2_CONFIG['output_width'];outH=round(H*outW/W)
raw_q2=OUT/'q2'/'highway_raw.mp4'
writer=cv2.VideoWriter(str(raw_q2),cv2.VideoWriter_fourcc(*'mp4v'),q2_info['fps'],(outW,outH))
assert writer.isOpened()
cap=cv2.VideoCapture(str(DATA/'highway.mp4'))
previous_gray=None
start=time.time()
try:
    for frame_index in tqdm(range(q2_info['frames']),desc='Q2: every highway frame'):
        ok,frame=cap.read();assert ok,f'Unexpected end at frame {frame_index}'
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        flow=None if previous_gray is None else dense_backward(gray,previous_gray,Q2_CONFIG['flow_width'])
        result=q2_detector.track(frame,persist=True,tracker=str(tracker_path),classes=[2,5,7],
                                 conf=Q2_CONFIG['confidence'],iou=Q2_CONFIG['nms_iou'],
                                 agnostic_nms=True,imgsz=Q2_CONFIG['imgsz'],retina_masks=True,
                                 device=DEVICE,verbose=False)[0]
        boxes=result.boxes.xyxy.cpu().numpy()
        classes=result.boxes.cls.cpu().numpy().astype(int)
        confidences=result.boxes.conf.cpu().numpy()
        ids=result.boxes.id.int().cpu().numpy() if result.boxes.id is not None else np.full(len(boxes),-1)
        polygons=result.masks.xy if result.masks is not None else [None]*len(boxes)
        canvas=frame.copy();entries=[]
        for box,cls,confidence,track_id,polygon in zip(boxes,classes,confidences,ids,polygons):
            track_id=int(track_id);class_name=result.names[int(cls)]
            mask,mask_method=polygon_mask(flow.shape[:2] if flow is not None else (H//2,W//2),
                                           box,polygon,(W,H))
            displacement,pixels=mean_motion(flow,mask)
            speed=displacement*q2_info['fps']
            center=(np.asarray(box[:2])+np.asarray(box[2:]))/2
            previous=q2_history[track_id][-1] if track_id>=0 and q2_history[track_id] else None
            center_displacement=(float(np.linalg.norm(center-np.array(previous[1:])))
                                 if previous and previous[0]==frame_index-1 else np.nan)
            if track_id>=0:
                q2_history[track_id].append((frame_index,float(center[0]),float(center[1])))
                if np.isfinite(speed):q2_ema[track_id]=speed if track_id not in q2_ema else .35*speed+.65*q2_ema[track_id]
                points=np.array([p[1:] for p in q2_history[track_id]],np.int32)
                if len(points)>1:cv2.polylines(canvas,[points],False,track_color(track_id),3,cv2.LINE_AA)
            shown=q2_ema.get(track_id,speed)
            color=track_color(track_id) if track_id>=0 else (180,180,180)
            draw_label(canvas,box,f'ID {track_id if track_id>=0 else "pending"} {class_name}\n{motion_label(shown,"px/s")}',color,scale=.65)
            q2_rows.append({'frame':frame_index,'time_s':frame_index/q2_info['fps'],'track_id':track_id,
                            'class':class_name,'confidence':float(confidence),
                            'x1':float(box[0]),'y1':float(box[1]),'x2':float(box[2]),'y2':float(box[3]),
                            'center_x':float(center[0]),'center_y':float(center[1]),
                            'flow_px_frame':displacement,'flow_px_s':speed,'display_ema_px_s':shown,
                            'center_displacement_px_frame':center_displacement,'mask_method':mask_method,
                            'valid_flow_pixels':pixels})
            entries.append({'id':track_id,'class':class_name,'conf':round(float(confidence),5),
                            'box':box.round(2).tolist(),
                            'polygon':np.asarray(polygon).round(1).tolist() if polygon is not None else None})
        cv2.rectangle(canvas,(0,0),(W,55),(15,15,15),-1)
        cv2.putText(canvas,f'Q2 | Frame {frame_index:04d}/{q2_info["frames"]-1} | 25 FPS | original-image px/s (EMA display)',
                    (18,37),cv2.FONT_HERSHEY_SIMPLEX,.9,(255,255,255),2,cv2.LINE_AA)
        writer.write(cv2.resize(canvas,(outW,outH),interpolation=cv2.INTER_AREA))
        if frame_index in [0,375,750,1125,1500]:
            q2_frames[frame_index]=canvas.copy()
            cv2.imwrite(str(OUT/'q2'/f'frame_{frame_index:04d}.jpg'),canvas)
        q2_frame_rows.append({'frame':frame_index,'detections':len(boxes),'tracked':int((ids>=0).sum())})
        q2_cache.append(entries)
        previous_gray=gray
finally:
    cap.release();writer.release()
q2_df=pd.DataFrame(q2_rows);q2_frame_df=pd.DataFrame(q2_frame_rows)
q2_df.to_csv(OUT/'q2'/'vehicle_motion.csv',index=False)
q2_frame_df.to_csv(OUT/'q2'/'frame_audit.csv',index=False)
with gzip.open(OUT/'q2'/'detections.json.gz','wt') as file:json.dump(q2_cache,file)
q2_output=OUT/'q2'/'highway_tracking_speed.mp4'
q2_output_info=encode_video(raw_q2,q2_output)
assert q2_output_info['frames']==q2_info['frames']==len(q2_frame_df)
assert abs(q2_output_info['fps']-25)<1e-6
assert np.allclose(q2_df.flow_px_s,q2_df.flow_px_frame*25,equal_nan=True)
assert set(q2_df['class'])<= {'car','truck','bus'}
q2_summary={'processed_frames':len(q2_frame_df),'output_frames':q2_output_info['frames'],
            'unique_vehicle_ids':int(q2_df.loc[q2_df.track_id>=0,'track_id'].nunique()),
            'vehicle_observations':len(q2_df),'unassigned_observations':int((q2_df.track_id<0).sum()),
            'invalid_motion_after_first_frame':int(q2_df.loc[q2_df.frame>0,'flow_px_frame'].isna().sum()),
            'elapsed_seconds':round(time.time()-start,1),'config':Q2_CONFIG,
            'motion_percentiles_px_s':q2_df.flow_px_s.quantile([.05,.5,.95]).to_dict()}
(OUT/'q2'/'summary.json').write_text(json.dumps(q2_summary,indent=2))
display(Markdown('### Q2 results and checks'))
print('Q2_SUMMARY',json.dumps(q2_summary))
display(q2_df.head(12))
display_contact({k:q2_frames[k] for k in [0,375,750,1500]},OUT/'q2'/'representative_frames.png',size=(15,9))
show_saved(OUT/'q2'/'representative_frames.png',width=1000)
fig,axes=plt.subplots(1,2,figsize=(13,4))
longest=q2_df[q2_df.track_id>=0].groupby('track_id').size().nlargest(6).index
for tid in longest:
    track=q2_df[q2_df.track_id==tid]
    axes[0].plot(track.time_s,track.flow_px_s,label=f'ID {tid}',alpha=.8)
axes[0].set(xlabel='Time (s)',ylabel='Mean foreground optical flow (original px/s)',title='Six longest vehicle tracks; raw estimates')
axes[0].legend(fontsize=8)
axes[1].plot(q2_frame_df.frame,q2_frame_df.tracked,lw=1)
axes[1].set(xlabel='Frame',ylabel='Tracked vehicles',title='Coverage audit: all 1501 frames')
fig.tight_layout();fig.savefig(OUT/'q2'/'motion_audit.png');plt.show();plt.close(fig)
show_saved(OUT/'q2'/'motion_audit.png')
display(Video(str(q2_output),embed=True,width=960))
print('Q2_CHECK passed: full video, vehicle-only classes, original-scale flow, and 25 FPS unit conversion.')

### Results and discussion

The output contains all 1,501 frames at 25 FPS. We recorded 8,268 vehicle observations across 44 confirmed tracker IDs, with an ID assigned to every stored observation. Flow measurements are available after frame 0. The 44 IDs describe the tracker's output, rather than a manually verified count of distinct vehicles.

Raw pixel speed has a median of 90.76 px/s; the 5th and 95th percentiles are 29.96 and 373.83 px/s. In the longest tracks, the curves generally rise as vehicles approach the camera. Perspective explains much of this pattern: the same road displacement covers more image pixels near the camera. These measurements therefore compare apparent image motion and cannot be read directly as road speeds.

The representative frames show boxes, class/ID labels, trajectories, and speed labels. Small vehicles near the horizon and overlapping vehicles remain difficult. Mask boundaries and box jitter can change the estimates, and occasional car/truck class changes do not necessarily mean the vehicle's ID changed. Foreground masks limit background contamination, and the displayed moving average reduces label flicker. Raw measurements and the track-lifetime audit remain available for closer inspection. Without ground-truth IDs or a physical calibration, we cannot assign a tracking-accuracy score or convert these values to km/h.

![Vehicle tracking at frame 750](../assets/traffic.jpg)

## 3. Person and Dog Tracking with Motion Analysis

### 3(a)-(b) Detection and tracking

YOLO11m-seg detects people and dogs on every frame. We follow the main person as ID 1 and the dog as ID 2. A class-aware Kalman filter predicts position, Lucas-Kanade feature motion helps bridge detections, and spatial and size checks associate candidates with each track.

SAM2.1 supplies temporal masks initialized automatically from the first-frame person and dog detections. Later correction prompts also come from the detector. A reliable dog-class detection takes priority over an uncalibrated temporal mask.

The dog becomes small and passes behind the person. A temporal mask can remain nonempty while switching to part of the person, so we check identity before accepting it. Temporal dog candidates are rejected when more than 20% of their foreground overlaps the person's mask or more than 35% of their box lies within the person's box. Near the person, we also reject boxes taller than 55% of the person or larger than 40% of the person's box area. A separate anchor from reliable YOLO detections limits temporal growth, preventing a sequence of slightly enlarged masks from gradually becoming a person-sized box.

If neither a valid mask nor a reliable detection is available, dense backward flow can transport the last accepted dog foreground. These proposals must pass the same identity checks. Local-context detections offer another recovery opportunity. When no candidate passes, we retain a small predicted box. During longer gaps, its offset and scale follow the person relative to the last accepted dog/person geometry, rather than expanding onto the human. A later dog detection can re-associate with ID 2. Fully hidden positions remain estimates.

### 3(c) Motion estimation

Motion is expressed in original-image px/frame, with px/s also saved using the video's 30 FPS. For accepted frame evidence, we average foreground dense-flow magnitudes. During prediction-only intervals, we use the displacement of the predicted box center: under the box-translation model, every point has that same displacement. These values are labeled EST in the video and recorded as `predicted_box_translation` in the CSV, separately from `foreground_dense_flow`. The sampled flow over a predicted region is retained only as a diagnostic. Frame 0 is unavailable because it has no preceding frame.

Camera motion, perspective, and body movement contribute to apparent image motion; the values are not calibrated running speeds.

### 3(d)-(e) Trajectory and occlusion handling

We save the dog's box center on every frame and distinguish four sources:

- **DET:** a YOLO detection.
- **SEG:** an accepted temporal segmentation mask.
- **FLOW:** accepted foreground propagation through measured flow.
- **PRED:** a predicted position without accepted target evidence.

Predicted boxes are dashed, and prediction segments are marked separately in the video. The continuous full-video trajectory includes these estimated positions. The motion plot marks predicted translation values separately. Four representative frames, a 17-frame contact sheet, and a focused review of frames 313, 330, 340, and 360 support the occlusion analysis.

The first code cell prepares detector/SAM outputs; its raw mask diagnostics precede the identity checks. The next cell produces the final boxes and trajectory. SAM masks are model estimates, and placeholder scores of one are not treated as confidence probabilities.

In [ ]:
from ultralytics.models.sam import SAM2VideoPredictor
import gc

SAM2_CONFIG={'weights':'sam2.1_b.pt','imgsz':1024,'revision':'dual-object-detector-anchors-v2',
             'anchor_interval':30,'dog_anchor_min_confidence':.12,
             'video_sha256':hashlib.sha256((DATA/'video.mp4').read_bytes()).hexdigest()}
sam_video_info=video_metadata['video.mp4']
SW,SH=sam_video_info['width'],sam_video_info['height']

# A complete detector pass is cached and reused by the final Q3 renderer.
# Person and dog detections remain separate during NMS.
q3_raw_detection_path=OUT/'q3'/'yolo_all_frame_detections.json.gz'
q3_raw_metadata_path=OUT/'q3'/'yolo_all_frame_detections_metadata.json'
detector_config={'weights':'yolo11m-seg.pt','imgsz':1280,'conf':.08,'agnostic_nms':False,
                 'video_sha256':SAM2_CONFIG['video_sha256']}
detector_cache_ok=(q3_raw_detection_path.exists() and q3_raw_metadata_path.exists() and
                   json.loads(q3_raw_metadata_path.read_text())==detector_config)
sam_initializer=YOLO('yolo11m-seg.pt')
if detector_cache_ok:
    with gzip.open(q3_raw_detection_path,'rt') as file:q3_yolo_cache=json.load(file)
else:
    q3_yolo_cache=[]
    cap=cv2.VideoCapture(str(DATA/'video.mp4'))
    try:
        for index in tqdm(range(sam_video_info['frames']),desc='Q3 detector: every frame'):
            ok,frame=cap.read();assert ok
            result=sam_initializer.predict(frame,classes=[0,16],conf=.08,imgsz=1280,iou=.6,
                                           agnostic_nms=False,retina_masks=True,device=DEVICE,verbose=False)[0]
            polygons=result.masks.xy if result.masks is not None else [None]*len(result.boxes)
            entries=[]
            for box,cls,score,poly in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.cls.cpu().numpy(),
                                         result.boxes.conf.cpu().numpy(),polygons):
                entries.append({'class':int(cls),'confidence':float(score),'box':box.tolist(),
                                'polygon':np.asarray(poly).tolist() if poly is not None else None,'source':'full-frame'})
            q3_yolo_cache.append(entries)
    finally:cap.release()
    with gzip.open(q3_raw_detection_path,'wt') as file:json.dump(q3_yolo_cache,file)
    q3_raw_metadata_path.write_text(json.dumps(detector_config,indent=2))
assert len(q3_yolo_cache)==sam_video_info['frames']
initial_boxes={cls:max([c for c in q3_yolo_cache[0] if c['class']==cls],key=lambda c:c['confidence'])['box']
               for cls in [0,16]}
SAM2_CONFIG['initial_person_box']=initial_boxes[0]
SAM2_CONFIG['initial_dog_box']=initial_boxes[16]
sam_cache_path=OUT/'q3'/'sam2_dog_masks.npz'
sam_meta_path=OUT/'q3'/'sam2_dog_masks_metadata.json'
cache_ok=(sam_cache_path.exists() and sam_meta_path.exists() and
          json.loads(sam_meta_path.read_text()).get('config')==SAM2_CONFIG)
sam_review_frames={};sam_anchor_log=[]

def extract_dog_mask(result):
    # The person is initialized first and stays visible in this supplied clip.
    # If the dog mask is absent, only the person remains after SAM filtering.
    if result.masks is not None and len(result.masks)>=2:
        mask=result.masks.data[1].detach().cpu().numpy().astype(bool)
        assert mask.shape==(SH,SW)
        return mask
    return np.zeros((SH,SW),bool)

if cache_ok:
    loaded=np.load(sam_cache_path)
    sam_dog_masks=np.unpackbits(loaded['packed'],axis=2)[:,:,:SW].astype(bool)
    sam_anchor_log=json.loads(sam_meta_path.read_text()).get('automatic_anchor_log',[])
    print('Using verified detector-anchored SAM2 mask cache:',sam_dog_masks.shape)
else:
    if 'strong_dog_detector' in globals():del strong_dog_detector
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    sam_video_predictor=SAM2VideoPredictor(overrides={'model':SAM2_CONFIG['weights'],'imgsz':1024,
        'conf':.01,'task':'segment','mode':'predict','device':DEVICE,'verbose':False,'save':False})
    packed_masks=[];last_dog_box=np.array(initial_boxes[16],float);last_anchor=0
    sam_stream=sam_video_predictor(source=str(DATA/'video.mp4'),
        bboxes=[initial_boxes[0],initial_boxes[16]],stream=True)
    for index,result in enumerate(tqdm(sam_stream,total=sam_video_info['frames'],desc='SAM2: dual targets + automatic anchors')):
        if index==0:
            # v8.3.221's correction API reads this option although SAM2Model
            # omits its constructor default. Use the standard non-conditional
            # correction policy explicitly.
            sam_video_predictor.model.add_all_frames_to_correct_as_cond=False
        frame=result.orig_img
        mask=extract_dog_mask(result)
        entries=q3_yolo_cache[index]
        person=max([c for c in entries if c['class']==0],key=lambda c:c['confidence'],default=None)
        person_mask=polygon_mask((SH,SW),person['box'],person['polygon'])[0] if person else np.zeros((SH,SW),bool)
        area=int(mask.sum())
        overlap=float((mask&person_mask).sum()/max(area,1))
        valid=area>=4 and overlap<=.8
        lm=np.r_[(last_dog_box[:2]+last_dog_box[2:])/2,last_dog_box[2:]-last_dog_box[:2]]
        anchors=[]
        for c in entries:
            if c['class']!=16 or c['confidence']<SAM2_CONFIG['dog_anchor_min_confidence']:continue
            box=np.array(c['box']);cm=np.r_[(box[:2]+box[2:])/2,box[2:]-box[:2]]
            distance=float(np.linalg.norm(cm[:2]-lm[:2]))
            ratio=float(cm[2]*cm[3]/max(lm[2]*lm[3],1))
            if distance<=max(35,2*float(lm[2:].max())) and .05<=ratio<=12:
                anchors.append((np.log(c['confidence'])-distance/max(35,float(lm[2:].max())),c))
        candidate=max(anchors,key=lambda item:item[0])[1] if anchors else None
        refresh=candidate is not None and index>0 and (not valid or index-last_anchor>=SAM2_CONFIG['anchor_interval'])
        if refresh:
            # Public SAM2 correction API. Box corners use top-left letterbox
            # coordinates and SAM labels 2/3, matching Ultralytics v8.3.221.
            scale=1024/max(SH,SW)
            corners=np.array(candidate['box'],dtype=np.float32).reshape(1,2,2)*scale
            points=torch.tensor(corners,dtype=torch.float32,device=DEVICE)
            labels=torch.tensor([[2,3]],dtype=torch.int32,device=DEVICE)
            corrected,_=sam_video_predictor.add_new_prompts(obj_id=1,points=points,labels=labels,
                                                           frame_idx=sam_video_predictor.dataset.frame)
            sam_video_predictor.propagate_in_video_preflight()
            # This pinned version flattens object/spatial dimensions twice in
            # add_new_prompts; restore the two-object square mask tensor.
            if corrected.ndim==2:
                corrected=corrected.reshape(2,corrected.shape[-1],corrected.shape[-1])
            assert corrected.ndim==3 and corrected.shape[0]==2
            scores=torch.ones(corrected.shape[0],dtype=corrected.dtype,device=corrected.device)
            revised=sam_video_predictor.postprocess((corrected,scores),
                sam_video_predictor.inference_state['im'],[frame])[0]
            mask=extract_dog_mask(revised)
            area=int(mask.sum());overlap=float((mask&person_mask).sum()/max(area,1))
            valid=area>=4 and overlap<=.8
            last_anchor=index
            last_dog_box=np.array(candidate['box'],float)
            sam_anchor_log.append({'frame':index,'confidence':candidate['confidence'],
                                   'box':candidate['box'],'reason':'invalid/overlapping mask or periodic refresh'})
        if valid:
            ys,xs=np.where(mask)
            last_dog_box=np.array([xs.min(),ys.min(),xs.max()+1,ys.max()+1],float)
        else:
            # Never cache the main person as positive dog frame evidence.
            mask[:]=False
        packed_masks.append(np.packbits(mask,axis=1))
        if index in [0,160,180,220,300,360,420,540,600,659]:
            preview=frame.copy()
            if mask.any():
                preview[mask]=(.6*preview[mask]+.4*np.array([0,255,255])).astype(np.uint8)
                ys,xs=np.where(mask)
                cv2.rectangle(preview,(int(xs.min()),int(ys.min())),(int(xs.max()),int(ys.max())),(0,255,255),1)
            sam_review_frames[index]=preview
        if index in [360,659]:
            path=OUT/'q3'/f'sam2_anchor_review_{index}.png';cv2.imwrite(str(path),sam_review_frames[index])
            show_saved(path,width=640)
    assert len(packed_masks)==sam_video_info['frames']
    packed_masks=np.stack(packed_masks)
    np.savez_compressed(sam_cache_path,packed=packed_masks)
    sam_dog_masks=np.unpackbits(packed_masks,axis=2)[:,:,:SW].astype(bool)
    del sam_video_predictor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
sam_mask_areas=sam_dog_masks.sum(axis=(1,2))
sam_summary={'config':SAM2_CONFIG,'processed_frames':len(sam_dog_masks),
             'nonempty_masks':int((sam_mask_areas>0).sum()),'masks_with_at_least_4_pixels':int((sam_mask_areas>=4).sum()),
             'automatic_anchor_log':sam_anchor_log,
             'note':'SAM2 masks are model estimates, not ground truth. Placeholder scores of one are not probabilities. All prompts are automatic detector outputs.'}
sam_meta_path.write_text(json.dumps(sam_summary,indent=2))
print('SAM2_MASK_SUMMARY',json.dumps({k:v for k,v in sam_summary.items() if k!='automatic_anchor_log'}))
print('Automatic detector anchors:',len(sam_anchor_log))
if sam_review_frames:
    display_contact(sam_review_frames,OUT/'q3'/'sam2_mask_contact.png',columns=3,size=(15,12))
    show_saved(OUT/'q3'/'sam2_mask_contact.png',width=1100)

In [ ]:
Q3_CONFIG={'detector':'yolo11m-seg.pt','full_imgsz':1280,'roi_imgsz':960,
           'confidence':.08,'association_min_confidence':.12,'roi_min_side':112,
           'prediction_warning_gap':90,'flow_width':640,'revision':'identity-gated-occlusion-v7',
           'temporal_person_mask_overlap_max':.20,'temporal_person_box_fraction_max':.35,
           'near_person_height_ratio_max':.55,'near_person_area_ratio_max':.40,
           'temporal_model':'sam2.1_b.pt'}
q3_detector=YOLO(Q3_CONFIG['detector'])
OBSERVED_STATUSES={'detected','mask_tracked','flow_tracked'}
assert len(sam_dog_masks)==video_metadata['video.mp4']['frames']
assert sam_summary['config']==SAM2_CONFIG, 'Temporal masks are stale or preparation failed.'
assert json.loads(sam_meta_path.read_text())['config']==SAM2_CONFIG
assert len(q3_yolo_cache)==video_metadata['video.mp4']['frames']

def clip_box(box,width,height):
    box=np.array(box,dtype=float)
    box[[0,2]]=np.clip(box[[0,2]],0,width-1)
    box[[1,3]]=np.clip(box[[1,3]],0,height-1)
    box[2]=min(width-1,max(box[0]+2,box[2]));box[3]=min(height-1,max(box[1]+2,box[3]))
    return box

def box_measurement(box):
    return np.r_[(box[:2]+box[2:])/2,box[2:]-box[:2]].astype(np.float32)

def measurement_box(measurement):
    cx,cy,w,h=np.asarray(measurement).reshape(-1)[:4]
    w,h=max(2,w),max(2,h)
    return np.array([cx-w/2,cy-h/2,cx+w/2,cy+h/2])

def box_iou(a,b):
    lo=np.maximum(a[:2],b[:2]);hi=np.minimum(a[2:],b[2:])
    inter=np.maximum(hi-lo,0).prod()
    return float(inter/max((a[2:]-a[:2]).prod()+(b[2:]-b[:2]).prod()-inter,1e-6))

def intersection_fraction(box,other):
    extent=np.maximum(np.minimum(box[2:],other[2:])-np.maximum(box[:2],other[:2]),0)
    return float(extent.prod()/max(np.prod(box[2:]-box[:2]),1))

def dog_identity_check(candidate,person_box,person_support,track):
    """Reject person-shaped dog evidence before association; never gate by frame number."""
    box=np.asarray(candidate['box'],float);cm=box_measurement(box)
    pm=box_measurement(person_box)
    region=(candidate['mask'] if candidate.get('temporal_mask',False) else
            polygon_mask(person_support.shape,box,candidate['polygon'])[0])
    mask_overlap=float((region&person_support).sum()/max(int(region.sum()),1))
    box_fraction=intersection_fraction(box,person_box)
    near=(abs(cm[0]-pm[0])<max(pm[2],cm[2]) and
          abs(cm[1]-pm[1])<pm[3])
    metrics={'person_mask_overlap':mask_overlap,'person_box_fraction':box_fraction,
             'dog_person_height_ratio':float(cm[3]/max(pm[3],1))}
    temporal=candidate.get('temporal_mask',False)
    if near and (cm[3]>Q3_CONFIG['near_person_height_ratio_max']*pm[3] or
                 cm[2]*cm[3]>Q3_CONFIG['near_person_area_ratio_max']*pm[2]*pm[3]):
        return False,'person-like box scale',metrics
    if box_fraction>(Q3_CONFIG['temporal_person_box_fraction_max'] if temporal else .65):
        return False,'inside person box',metrics
    if temporal and mask_overlap>Q3_CONFIG['temporal_person_mask_overlap_max']:
        return False,'person foreground overlap',metrics
    if temporal and track.detector_reference is not None:
        reference_size,reference_person_height=track.detector_reference
        expected=reference_size*np.clip(pm[3]/max(reference_person_height,1),.5,2.)
        if cm[2]>max(18,2.8*expected[0]) or cm[3]>max(22,2.*expected[1]):
            return False,'growth beyond independent detector anchor',metrics
    return True,'accepted',metrics

def result_candidates(result,offset=(0,0),source='full-frame'):
    candidates=[]
    polygons=result.masks.xy if result.masks is not None else [None]*len(result.boxes)
    shift=np.array(offset)
    for box,cls,conf,poly in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.cls.cpu().numpy(),
                               result.boxes.conf.cpu().numpy(),polygons):
        candidates.append({'class':int(cls),'confidence':float(conf),
                           'box':box.astype(float)+np.tile(shift,2),
                           'polygon':np.asarray(poly)+shift if poly is not None else None,
                           'source':source})
    return candidates

class ClassAwareTrack:
    """One class-constrained target track with Kalman/LK prediction and gated association.
    The supplied clip contains one main person and one dog. Low-confidence distant
    duplicates never create extra identities. IDs are initialized once and associated
    using spatial/scale consistency; missing observations remain explicitly predicted.
    """
    def __init__(self,class_id,name,track_id):
        self.class_id,self.name,self.track_id=class_id,name,track_id
        self.box=None;self.mask=None;self.misses=0;self.initialized=False;self.prediction=None
        self.last_observed_box=None
        self.flow_probability=None
        self.detector_reference=None
        self.person_context=None;self.last_person_context=None
        self.kf=cv2.KalmanFilter(8,4)
        self.kf.transitionMatrix=np.eye(8,dtype=np.float32)
        self.kf.transitionMatrix[:4,4:]=np.eye(4,dtype=np.float32)
        # Propagate center velocity, but never extrapolate a shrinking size to zero.
        self.kf.transitionMatrix[2,6]=0
        self.kf.transitionMatrix[3,7]=0
        self.kf.measurementMatrix=np.c_[np.eye(4),np.zeros((4,4))].astype(np.float32)
        self.kf.processNoiseCov=np.diag([.1,.1,.2,.2,.04,.04,.04,.04]).astype(np.float32)
        self.kf.measurementNoiseCov=np.diag([1.,1.,2.,2.]).astype(np.float32)
        self.kf.errorCovPost=np.eye(8,dtype=np.float32)*10

    def predict(self,previous_gray,gray):
        if not self.initialized:return None
        predicted=self.kf.predict().reshape(-1)[:4]
        observed=box_measurement(self.last_observed_box)
        predicted[2:4]=observed[2:4]
        if self.misses>=4:
            # After an extended miss, hold the last reliable search anchor rather
            # than letting constant-velocity prediction drift into the background.
            predicted[:2]=observed[:2]
            if self.name=='dog' and self.person_context is not None and self.last_person_context is not None:
                # During full occlusion, follow scene/person motion while retaining
                # the last reliable relative offset and a small dog-sized box.
                now=box_measurement(self.person_context)
                anchor=box_measurement(self.last_person_context)
                scale=float(np.clip(now[3]/max(anchor[3],1),.75,1.4))
                predicted[:2]=now[:2]+(observed[:2]-anchor[:2])*scale
                predicted[2:4]=observed[2:4]*scale
        self.lk_points=0
        if previous_gray is not None and self.mask is not None and self.misses<=2:
            points=cv2.goodFeaturesToTrack(previous_gray,maxCorners=60,qualityLevel=.015,minDistance=2,
                                          mask=self.mask.astype(np.uint8)*255,blockSize=3)
            if points is not None and len(points)>=4:
                nxt,ok,_=cv2.calcOpticalFlowPyrLK(previous_gray,gray,points,None,winSize=(21,21),maxLevel=3)
                back,ok_back,_=cv2.calcOpticalFlowPyrLK(gray,previous_gray,nxt,None,winSize=(21,21),maxLevel=3)
                good=(ok.ravel()>0)&(ok_back.ravel()>0)&(np.linalg.norm(points[:,0]-back[:,0],axis=1)<1.5)
                self.lk_points=int(good.sum())
                if good.sum()>=4:
                    displacement=np.median(nxt[good,0]-points[good,0],axis=0)
                    if np.linalg.norm(displacement)<45:
                        predicted[:2]=box_measurement(self.box)[:2]+displacement
                        self.kf.statePre[:2,0]=predicted[:2]
        self.prediction=clip_box(measurement_box(predicted),gray.shape[1],gray.shape[0])
        return self.prediction

    def associate(self,candidates,gray):
        choices=[c for c in candidates if c['class']==self.class_id and
                 (c.get('temporal_mask',False) or c['confidence']>=Q3_CONFIG['association_min_confidence'])]
        chosen=None
        if not self.initialized:
            if choices:chosen=max(choices,key=lambda c:c['confidence'] if np.isfinite(c['confidence']) else -1)
        else:
            pm=box_measurement(self.prediction)
            gate=min(100,max(18,.85*np.hypot(pm[2],pm[3]))*np.sqrt(1+.2*self.misses))
            ranked=[]
            for c in choices:
                cm=box_measurement(c['box']);distance=float(np.linalg.norm(cm[:2]-pm[:2]))
                anchor_size=box_measurement(self.last_observed_box)[2:4]
                ratio=(cm[2]*cm[3])/max(anchor_size.prod(),1)
                min_ratio,max_ratio=(.04,20) if self.misses>=4 else (.12,7)
                if distance<=gate and min_ratio<=ratio<=max_ratio:
                    # Prefer a reliable class detection over an uncalibrated temporal mask.
                    evidence_score=(0.0 if c.get('temporal_mask',False) else
                                    1.5+np.log(max(c['confidence'],1e-6)))
                    score=evidence_score+1.5*box_iou(self.prediction,c['box'])-distance/gate
                    ranked.append((score,c))
            if ranked:chosen=max(ranked,key=lambda item:item[0])[1]
        if chosen is not None:
            measurement=box_measurement(chosen['box'])
            if not self.initialized:
                self.kf.statePost=np.r_[measurement,np.zeros(4)].astype(np.float32).reshape(8,1)
                self.initialized=True
            else:self.kf.correct(measurement.reshape(4,1))
            self.box=clip_box(chosen['box'],gray.shape[1],gray.shape[0])
            self.last_observed_box=self.box.copy()
            if self.person_context is not None:self.last_person_context=self.person_context.copy()
            self.kf.statePost[6:8,0]=0
            self.mask=(chosen['mask'].astype(bool) if chosen.get('temporal_mask',False) else
                       polygon_mask(gray.shape,self.box,chosen['polygon'])[0])
            self.flow_probability=chosen.get('probability',self.mask.astype(np.float32))
            self.misses=0
            return chosen,('flow_tracked' if chosen.get('flow_mask',False) else
                           ('mask_tracked' if chosen.get('temporal_mask',False) else 'detected'))
        if self.initialized:
            self.misses+=1;self.box=self.prediction
            self.mask,_=polygon_mask(gray.shape,self.box)
            self.flow_probability=None
            return None,'predicted' if self.misses<=Q3_CONFIG['prediction_warning_gap'] else 'uncertain'
        return None,'uninitialized'

q3_info=video_metadata['video.mp4'];W3,H3=q3_info['width'],q3_info['height']
flow_grid_y,flow_grid_x=np.indices((H3,W3),dtype=np.float32)
tracks=[ClassAwareTrack(0,'person',1),ClassAwareTrack(16,'dog',2)]
q3_rows,q3_frame_rows,q3_detection_cache=[],[],[]
q3_frames={};q3_history={'person':[],'dog':[]}
q3_raw=OUT/'q3'/'person_dog_raw.mp4'
writer=cv2.VideoWriter(str(q3_raw),cv2.VideoWriter_fourcc(*'mp4v'),q3_info['fps'],(W3,H3))
assert writer.isOpened()
cap=cv2.VideoCapture(str(DATA/'video.mp4'));previous_gray=None;start=time.time()
q3_selected=[0,160,180,200,220,240,270,300,313,330,340,360,420,480,540,600,659]
q3_rejection_rows=[]
try:
    for frame_index in tqdm(range(q3_info['frames']),desc='Q3: person/dog with ROI recovery'):
        ok,frame=cap.read();assert ok,f'Unexpected end at frame {frame_index}'
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        flow=None if previous_gray is None else dense_backward(gray,previous_gray,Q3_CONFIG['flow_width'])
        for track in tracks:
            if track.name=='dog':track.person_context=tracks[0].prediction
            track.predict(previous_gray,gray)
        # The preparation cell already detected every frame; reuse the exact cache.
        candidates=[{**c,'box':np.array(c['box'],float),
                     'polygon':np.array(c['polygon']) if c['polygon'] is not None else None}
                    for c in q3_yolo_cache[frame_index]]
        full_detection_count=len(candidates)
        dog_track=tracks[1]
        frame_rejections=[]
        sam_mask=sam_dog_masks[frame_index]
        sam_area=int(sam_mask.sum())
        sam_mask_available=4<=sam_area<=25000
        sam_person_overlap=0.0
        person_proposals=[c for c in candidates if c['class']==0]
        person_support=np.zeros(gray.shape,bool)
        person_box=(tracks[0].prediction if tracks[0].initialized else None)
        if person_proposals:
            person_proposal=max(person_proposals,key=lambda c:c['confidence'])
            person_box=np.asarray(person_proposal['box'],float)
            person_support=polygon_mask(gray.shape,person_proposal['box'],person_proposal['polygon'])[0]
        if sam_mask_available:
            sam_person_overlap=float((sam_mask&person_support).sum()/max(sam_area,1))
            if sam_person_overlap>.8:
                sam_mask_available=False
        if sam_mask_available:
            sy,sx=np.where(sam_mask)
            sam_box=np.array([sx.min(),sy.min(),sx.max()+1,sy.max()+1],float)
            contours,_=cv2.findContours(sam_mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            contour=max(contours,key=cv2.contourArea)[:,0,:] if contours else None
            # The SAM2 implementation returns placeholder scores of one. Do not
            # present them as dog-class confidence or calibrated mask confidence.
            candidates.append({'class':16,'confidence':np.nan,'box':sam_box,'polygon':contour,
                               'source':'SAM2 temporal mask','temporal_mask':True,'mask':sam_mask})
        # All temporal and semantic candidates must pass the same identity checks.
        def filter_dog_candidates(items):
            kept=[]
            for candidate in items:
                if candidate['class']!=16:
                    kept.append(candidate);continue
                accepted,reason,metrics=dog_identity_check(candidate,person_box,person_support,dog_track)
                candidate.update(metrics)
                if accepted:kept.append(candidate)
                else:
                    entry={'frame':frame_index,'source':candidate['source'],'reason':reason,
                           **metrics,'x1':float(candidate['box'][0]),'y1':float(candidate['box'][1]),
                           'x2':float(candidate['box'][2]),'y2':float(candidate['box'][3])}
                    frame_rejections.append(entry);q3_rejection_rows.append(entry)
            return kept
        candidates=filter_dog_candidates(candidates)
        sam_mask_available=any(c.get('source')=='SAM2 temporal mask' for c in candidates)
        used_roi=False
        flow_recovery_used=False
        reliable_dog_detection=False
        if dog_track.initialized:
            predicted_center=box_measurement(dog_track.prediction)[:2]
            reliable_dog_detection=any(c['class']==16 and not c.get('temporal_mask',False) and
                c['confidence']>=Q3_CONFIG['association_min_confidence'] and
                np.linalg.norm(box_measurement(c['box'])[:2]-predicted_center)<60 for c in candidates)
        if dog_track.initialized and not sam_mask_available and not reliable_dog_detection and flow is not None and dog_track.flow_probability is not None:
            # Advect the last accepted foreground through measured image flow.
            # Preserve subpixel probability transport instead of rounding a tiny
            # object's fractional motion to zero at each frame.
            probability=cv2.remap(dog_track.flow_probability.astype(np.float32),
                flow_grid_x+flow[...,0],flow_grid_y+flow[...,1],cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_CONSTANT,borderValue=0)
            if probability.max()>.05:
                probability/=probability.max()
                proposed=probability>.45
                person_fraction=float((proposed&person_support).sum()/max(int(proposed.sum()),1))
                probability[person_support]=0
                proposed=probability>.45
                if 4<=int(proposed.sum())<=25000 and person_fraction<=.8:
                    fy,fx=np.where(proposed)
                    flow_box=np.array([fx.min(),fy.min(),fx.max()+1,fy.max()+1],float)
                    contours,_=cv2.findContours(proposed.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                    poly=max(contours,key=cv2.contourArea)[:,0,:] if contours else None
                    candidates.append({'class':16,'confidence':np.nan,'box':flow_box,'polygon':poly,
                        'source':'Dense flow propagation','temporal_mask':True,'flow_mask':True,
                        'mask':proposed,'probability':probability})
                    flow_recovery_used=True
        if dog_track.initialized and not sam_mask_available and not flow_recovery_used and not reliable_dog_detection:
            pm=box_measurement(dog_track.prediction)
            near=[c for c in candidates if c['class']==16 and
                  np.linalg.norm(box_measurement(c['box'])[:2]-pm[:2])<max(25,pm[2:4].max())]
            if pm[2:4].max()<80 or not near or max(c['confidence'] for c in near)<.5:
                side=min(320,max(Q3_CONFIG['roi_min_side'],4*float(pm[2:4].max())))
                x1=max(0,int(pm[0]-side/2));y1=max(0,int(pm[1]-side/2))
                x2=min(W3,int(pm[0]+side/2));y2=min(H3,int(pm[1]+side/2))
                if x2-x1>20 and y2-y1>20:
                    roi=q3_detector.predict(frame[y1:y2,x1:x2],classes=[16],conf=.08,iou=.6,
                                             imgsz=Q3_CONFIG['roi_imgsz'],retina_masks=True,
                                             device=DEVICE,verbose=False)[0]
                    candidates.extend(result_candidates(roi,(x1,y1),'ROI'));used_roi=True
                if dog_track.misses>=3:
                    # Include the person/last-dog context during overlap recovery.
                    context=np.r_[np.minimum(dog_track.last_observed_box[:2],tracks[0].prediction[:2]),
                                  np.maximum(dog_track.last_observed_box[2:],tracks[0].prediction[2:])]
                    cm=box_measurement(context)
                    side=min(320,max(160,1.6*float(cm[2:4].max())))
                    rx1=max(0,int(cm[0]-side/2));ry1=max(0,int(cm[1]-side/2))
                    rx2=min(W3,int(cm[0]+side/2));ry2=min(H3,int(cm[1]+side/2))
                    if rx2-rx1>20 and ry2-ry1>20:
                        recovery=q3_detector.predict(frame[ry1:ry2,rx1:rx2],classes=[16],conf=.08,iou=.6,
                                                      imgsz=1280,retina_masks=True,device=DEVICE,verbose=False)[0]
                        candidates.extend(result_candidates(recovery,(rx1,ry1),'context ROI'))
        # Apply the checks to newly generated flow/ROI proposals too.
        existing_ids={id(c) for c in candidates if 'person_box_fraction' in c or c['class']!=16}
        candidates=[c for c in candidates if id(c) in existing_ids]+filter_dog_candidates(
            [c for c in candidates if id(c) not in existing_ids])
        direct_anchors=[c for c in candidates if c['class']==16 and
                        not c.get('temporal_mask',False) and c['confidence']>=.35]
        if direct_anchors:
            anchor=max(direct_anchors,key=lambda c:c['confidence'])
            if (not dog_track.initialized or
                np.linalg.norm(box_measurement(anchor['box'])[:2]-box_measurement(dog_track.prediction)[:2])<80):
                dog_track.detector_reference=(box_measurement(anchor['box'])[2:4],box_measurement(person_box)[3])
        current=[]
        for track in tracks:
            chosen,status=track.associate(candidates,gray)
            assert track.initialized,f'{track.name} not initialized at frame {frame_index}'
            current.append((track,chosen,status))
        canvas=frame.copy();cache=[]
        for track,chosen,status in current:
            visible_mask=track.mask.copy()
            if track.name=='dog':visible_mask &= ~tracks[0].mask
            displacement,pixels=mean_motion(flow,visible_mask)
            center=box_measurement(track.box)[:2]
            history=q3_history[track.name]
            center_motion=float(np.linalg.norm(center-np.array(history[-1][1:3]))) if history else np.nan
            raw_flow=displacement
            observed=status in OBSERVED_STATUSES
            if previous_gray is None:
                motion_method='unavailable_first_frame'
            elif not observed:
                # Mean translation of a predicted rigid box: every box point has
                # the same predicted center displacement. It is not measured flow.
                displacement=center_motion
                motion_method='predicted_box_translation'
            else:motion_method='foreground_dense_flow'
            history.append((frame_index,float(center[0]),float(center[1]),status))
            color=(0,0,255) if track.name=='person' else (0,255,255)
            for previous,current_point in zip(history[:-1],history[1:]):
                if track.name=='dog' or previous[0]>frame_index-60:
                    if current_point[3] in OBSERVED_STATUSES and previous[3] in OBSERVED_STATUSES:
                        cv2.line(canvas,tuple(np.rint(previous[1:3]).astype(int)),
                                 tuple(np.rint(current_point[1:3]).astype(int)),color,2,cv2.LINE_AA)
                    elif current_point[0]%3==0:
                        cv2.line(canvas,tuple(np.rint(previous[1:3]).astype(int)),
                                 tuple(np.rint(current_point[1:3]).astype(int)),(0,165,255),1,cv2.LINE_AA)
            suffix=('DET' if status=='detected' else ('SEG' if status=='mask_tracked' else
                    ('FLOW' if status=='flow_tracked' else f'PRED {track.misses}')))
            text=f'{track.name} ID {track.track_id} {suffix}\n{motion_label(displacement,"px/f")}'
            if not observed:text+=' EST'
            # Separate labels when the subjects overlap in the image.
            if track.name=='dog' and box_iou(track.box,tracks[0].box)>.01:
                cv2.line(canvas,tuple(np.rint(center).astype(int)),(min(W3-5,int(center[0]+70)),int(center[1])),color,1)
                draw_label(canvas,track.box,text,color,predicted=not observed,scale=.42,label_offset=(70,0))
            else:draw_label(canvas,track.box,text,color,predicted=not observed,scale=.42)
            q3_rows.append({'frame':frame_index,'time_s':frame_index/q3_info['fps'],
                            'class':track.name,'track_id':track.track_id,'status':status,
                            'source':chosen['source'] if chosen is not None else 'Kalman/LK prediction',
                            'confidence':chosen['confidence'] if chosen is not None else np.nan,
                            'gap_frames':track.misses,'lk_points':getattr(track,'lk_points',0),
                            'x1':float(track.box[0]),'y1':float(track.box[1]),
                            'x2':float(track.box[2]),'y2':float(track.box[3]),
                            'center_x':float(center[0]),'center_y':float(center[1]),
                            'mean_displacement_px_frame':displacement,
                            'mean_displacement_px_s':displacement*q3_info['fps'],
                            'center_displacement_px_frame':center_motion,'flow_pixels':pixels,
                            'motion_is_prediction_region':not observed,'motion_method':motion_method,
                            'sampled_region_flow_px_frame':raw_flow})
            q3_rows[-1].update({key:(chosen.get(key,np.nan) if chosen else np.nan) for key in
                              ['person_mask_overlap','person_box_fraction','dog_person_height_ratio']})
            cache.append({'class':track.name,'id':track.track_id,'status':status,
                          'box':track.box.round(2).tolist(),'confidence':chosen['confidence'] if chosen and np.isfinite(chosen['confidence']) else None,
                          'source':chosen['source'] if chosen else None,
                          'polygon':chosen['polygon'].round(2).tolist() if chosen and chosen['polygon'] is not None else None})
        cv2.rectangle(canvas,(0,0),(W3,28),(15,15,15),-1)
        cv2.putText(canvas,f'Q3 | {frame_index:03d}/659 | px/f | DET/SEG/FLOW | dashed:prediction',
                    (7,19),cv2.FONT_HERSHEY_SIMPLEX,.44,(255,255,255),1,cv2.LINE_AA)
        writer.write(canvas)
        if frame_index in q3_selected:
            q3_frames[frame_index]=canvas.copy()
            cv2.imwrite(str(OUT/'q3'/f'frame_{frame_index:04d}.jpg'),canvas)
        q3_frame_rows.append({'frame':frame_index,'full_detections':full_detection_count,'roi_used':used_roi,
                              'sam_mask_candidate_valid':sam_mask_available,'sam_person_overlap':sam_person_overlap,
                              'flow_recovery_used':flow_recovery_used})
        q3_frame_rows[-1]['rejected_dog_proposals']=len(frame_rejections)
        q3_detection_cache.append(cache);previous_gray=gray
finally:
    cap.release();writer.release()
q3_df=pd.DataFrame(q3_rows);q3_frame_df=pd.DataFrame(q3_frame_rows)
q3_df.to_csv(OUT/'q3'/'person_dog_motion.csv',index=False)
q3_frame_df.to_csv(OUT/'q3'/'frame_audit.csv',index=False)
pd.DataFrame(q3_rejection_rows).to_csv(OUT/'q3'/'rejected_dog_proposals.csv',index=False)
with gzip.open(OUT/'q3'/'associated_detections.json.gz','wt') as file:json.dump(q3_detection_cache,file)
q3_output=OUT/'q3'/'person_dog_tracking.mp4'
q3_output_info=encode_video(q3_raw,q3_output)
assert q3_output_info['frames']==q3_info['frames']==len(q3_frame_df)
assert len(q3_df)==2*q3_info['frames']
assert q3_df.groupby('class').track_id.nunique().eq(1).all()
assert q3_df.groupby('class').frame.nunique().eq(q3_info['frames']).all()
q3_summary={'processed_frames':len(q3_frame_df),'output_frames':q3_output_info['frames'],
            'ids':q3_df.groupby('class').track_id.first().to_dict(),
            'detected_frames':q3_df[q3_df.status=='detected'].groupby('class').size().to_dict(),
            'mask_tracked_frames':q3_df[q3_df.status=='mask_tracked'].groupby('class').size().to_dict(),
            'flow_tracked_frames':q3_df[q3_df.status=='flow_tracked'].groupby('class').size().to_dict(),
            'visually_supported_frames':q3_df[q3_df.status.isin(OBSERVED_STATUSES)].groupby('class').size().to_dict(),
            'predicted_frames':q3_df[~q3_df.status.isin(OBSERVED_STATUSES)].groupby('class').size().to_dict(),
            'max_prediction_gap_frames':q3_df.groupby('class').gap_frames.max().to_dict(),
            'roi_recovered_dog_frames':int(((q3_df['class']=='dog')&(q3_df.source.isin(['ROI','context ROI']))).sum()),
            'motion_unavailable_frames':q3_df[q3_df.mean_displacement_px_frame.isna()].groupby('class').size().to_dict(),
            'elapsed_seconds':round(time.time()-start,1),'config':Q3_CONFIG}
q3_summary['rejected_dog_proposals']=len(q3_rejection_rows)
q3_summary['review_frames']=[313,330,340,360]
q3_summary['motion_methods']=q3_df.groupby(['class','motion_method']).size().to_dict()
q3_summary['motion_methods']={f'{key[0]}:{key[1]}':int(value) for key,value in q3_summary['motion_methods'].items()}
(OUT/'q3'/'summary.json').write_text(json.dumps(q3_summary,indent=2))
display(Markdown('### Q3 results and checks'))
print('Q3_SUMMARY',json.dumps(q3_summary))
display(q3_df.groupby(['class','status']).agg(frames=('frame','size'),
            mean_motion_px_frame=('mean_displacement_px_frame','mean'),max_gap=('gap_frames','max')))
display_contact({k:q3_frames[k] for k in [0,180,270,659]},OUT/'q3'/'representative_frames.png',size=(15,9))
show_saved(OUT/'q3'/'representative_frames.png',width=1000)
display_contact(q3_frames,OUT/'q3'/'tracking_contact_sheet.png',columns=3,size=(15,17))
show_saved(OUT/'q3'/'tracking_contact_sheet.png',width=1000)
fig,axes=plt.subplots(2,2,figsize=(12,10))
for ax,index in zip(axes.flat,[313,330,340,360]):
    pair=q3_df[q3_df.frame==index]
    bounds=np.array([pair.x1.min(),pair.y1.min(),pair.x2.max(),pair.y2.max()])
    x1,y1=np.maximum(bounds[:2]-55,0).astype(int)
    x2,y2=np.minimum(bounds[2:]+55,[W3,H3]).astype(int)
    raw_review=read_frame(DATA/'video.mp4',index)
    ax.imshow(cv2.cvtColor(raw_review[y1:y2,x1:x2],cv2.COLOR_BGR2RGB))
    for _,record in pair.iterrows():
        ax.add_patch(plt.Rectangle((record.x1-x1,record.y1-y1),record.x2-record.x1,
            record.y2-record.y1,fill=False,edgecolor='red' if record['class']=='person' else 'gold',
            linewidth=2,linestyle='--' if record.status=='predicted' else '-'))
    status=pair[pair['class']=='dog'].status.iloc[0]
    ax.set_title(f'Frame {index} | dog: {status}');ax.axis('off')
fig.suptitle('Red: person ID 1 | Yellow: dog ID 2 | Dashed: predicted position',fontsize=12)
fig.tight_layout(rect=[0,0,1,.96]);fig.savefig(OUT/'q3'/'occlusion_review.png');plt.show();plt.close(fig)
show_saved(OUT/'q3'/'occlusion_review.png',width=1000)
dog=q3_df[q3_df['class']=='dog'].copy()
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
axes[0].imshow(cv2.cvtColor(read_frame(DATA/'video.mp4',0),cv2.COLOR_BGR2RGB),alpha=.65)
axes[0].plot(dog.center_x,dog.center_y,color='orange',lw=1,alpha=.7)
points=axes[0].scatter(dog.center_x,dog.center_y,c=dog.time_s,cmap='plasma',s=5)
axes[0].scatter(dog.center_x.iloc[0],dog.center_y.iloc[0],c='lime',marker='o',s=65,label='First frame')
axes[0].scatter(dog.center_x.iloc[-1],dog.center_y.iloc[-1],c='red',marker='x',s=65,label='Last frame')
axes[0].set(title='Dog trajectory: all 660 frames',xlabel='Image x (original px)',ylabel='Image y (original px)')
axes[0].legend(fontsize=8);fig.colorbar(points,ax=axes[0],label='Time (s)',fraction=.045)
for name,color in [('person','red'),('dog','goldenrod')]:
    part=q3_df[q3_df['class']==name]
    axes[1].plot(part.time_s,part.mean_displacement_px_frame,color=color,lw=.8,label=name)
    axes[2].plot(part.time_s,part.gap_frames,color=color,lw=1,label=name)
predicted=dog[~dog.status.isin(OBSERVED_STATUSES)]
axes[1].scatter(predicted.time_s,predicted.mean_displacement_px_frame,s=7,c='darkorange',label='dog: predicted box translation')
axes[1].set(title='Motion: foreground flow / predicted translation',xlabel='Time (s)',ylabel='Original px/frame')
axes[1].legend()
axes[2].set(title='Visibility and prediction audit',xlabel='Time (s)',ylabel='Frames without accepted image evidence')
axes[2].legend()
fig.tight_layout();fig.savefig(OUT/'q3'/'trajectory_motion_audit.png');plt.show();plt.close(fig)
show_saved(OUT/'q3'/'trajectory_motion_audit.png',width=1200)
display(Video(str(q3_output),embed=True,width=960))
print('Q3_CHECK passed: 660 frames, one persistent ID per target, full dog trajectory, and explicit predicted intervals.')

### Results and occlusion analysis

The revised video contains all 660 frames at 30 FPS, with person ID 1 and dog ID 2 in the stored sequence.

| Target | DET | SEG | FLOW | PRED |
|---|---:|---:|---:|---:|
| Person | 660 | 0 | 0 | 0 |
| Dog | 377 | 37 | 183 | 63 |

The dog has accepted detection, segmentation, or flow support on 597 of 660 frames (90.5%). The remaining 63 frames form one prediction interval, frames 285-347, lasting 2.1 seconds. A dog detection re-associates with ID 2 at frame 348. Motion estimates are available for both targets after frame 0; prediction-only motion is explicitly distinguished from measured foreground flow.

The targeted review confirmed a problem in the preceding version: at frames 330 and 340, SAM-derived dog boxes expanded over the person. A fixed ID and the earlier 97.6% evidence figure did not demonstrate correct target identity. The revised checks reject 94 dog proposals across the video. The yellow boxes at frames 313, 330, and 340 are now small, dashed predictions; frame 360 shows a dog detection again. At frames 330 and 340, the dog boxes are about 20 pixels tall, compared with the preceding version's 74-76 pixel boxes over the person.

We reviewed the focused occlusion figure and the wider contact sheet through approach, overlap, reappearance, and the final distant position. Reliable detector observations now receive priority, while masks and propagated foreground must pass the same person-overlap and scale checks. The reported coverage is lower because suspicious support is no longer counted as reliable evidence. This is preferable to an apparently complete trajectory that follows the wrong object.

The trajectory spans the first and last frames, but the hidden interval is an estimated path rather than a recovered ground-truth path. Its predicted position follows nearby scene/person movement and is corrected when the dog is detected again. Pixel motion also contains camera motion, perspective, and non-rigid movement. Without ground-truth boxes and IDs, these visual and consistency checks cannot establish perfect association accuracy or exact hidden locations.

![Tracking through occlusion](../assets/occlusion.jpg)

## Output checks and result files

The checks below confirm figure counts, complete frame sequences, encoded frame rates, motion-unit conversions, and per-track records. They also check that accepted temporal dog evidence passes the identity gates and that the reviewed overlap frames contain small dog-sized boxes. Prediction intervals and rejected proposals remain available for inspection. The download controls provide the figures, CSV measurements, summaries, and annotated videos.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED


assert len(feature_stats)==12 and len(prediction_stats)==4
assert len(list((OUT/'q1').glob('*_16channels.png')))==12
assert len(list((OUT/'q1').glob('*_gradcam.png')))==4
assert len(list((OUT/'q1').glob('*_channel_cams.png')))==4
assert q2_frame_df.frame.tolist()==list(range(1501))
assert q3_frame_df.frame.tolist()==list(range(660))
assert video_info(q2_output)['frames']==1501
assert video_info(q3_output)['frames']==660
assert abs(video_info(q2_output)['fps']-25)<1e-6
assert abs(video_info(q3_output)['fps']-30)<1e-6
assert np.allclose(q2_df.flow_px_s,q2_df.flow_px_frame*25,equal_nan=True)
assert np.allclose(q3_df.mean_displacement_px_s,q3_df.mean_displacement_px_frame*30,equal_nan=True)
assert (q2_df.flow_px_frame.dropna()>=0).all()
assert (q3_df.mean_displacement_px_frame.dropna()>=0).all()
assert q3_df.groupby('class').frame.nunique().eq(660).all()
assert q3_df.groupby('class').track_id.nunique().eq(1).all()

track_audit=[]
for tid,part in q2_df[q2_df.track_id>=0].groupby('track_id'):
    part=part.sort_values('frame')
    dt=part.frame.diff()
    movement=np.hypot(part.center_x.diff(),part.center_y.diff())/dt
    track_audit.append({'id':int(tid),'observations':len(part),'first_frame':int(part.frame.min()),
                        'last_frame':int(part.frame.max()),'max_detection_gap':int(dt.max()-1) if len(part)>1 else 0,
                        'max_center_motion_px_frame':float(movement.max()) if len(part)>1 else None})
track_audit=pd.DataFrame(track_audit)
track_audit.to_csv(OUT/'q2'/'track_lifetime_audit.csv',index=False)
dog_records=q3_df[q3_df['class']=='dog'].copy()
# Validate target identity checks instead of rewarding a high evidence count.
temporal_dog=dog_records[dog_records.status.isin(['mask_tracked','flow_tracked'])]
assert temporal_dog.person_box_fraction.le(Q3_CONFIG['temporal_person_box_fraction_max']+1e-6).all()
assert temporal_dog.person_mask_overlap.le(Q3_CONFIG['temporal_person_mask_overlap_max']+1e-6).all()
for index in [313,330,340,360]:
    d=dog_records[dog_records.frame==index].iloc[0]
    p=q3_df[(q3_df['class']=='person')&(q3_df.frame==index)].iloc[0]
    assert (d.y2-d.y1)<.55*(p.y2-p.y1), f'Person-sized dog box at review frame {index}'
missing=~dog_records.status.isin(['detected','mask_tracked','flow_tracked'])
groups=missing.ne(missing.shift()).cumsum()
if missing.any():
    gap_table=dog_records[missing].groupby(groups[missing]).agg(
        first_frame=('frame','min'),last_frame=('frame','max'),frames=('frame','size'))
else:gap_table=pd.DataFrame(columns=['first_frame','last_frame','frames'])
gap_table.to_csv(OUT/'q3'/'dog_prediction_intervals.csv',index=False)
checks={'q1_feature_grids':12,'q1_aggregate_gradcams':4,'q1_channel_cam_grids':4,
        'q2_processed_frames':1501,'q2_fps':25,'q3_processed_frames':660,'q3_fps':30,
        'q3_ids_per_class':q3_df.groupby('class').track_id.nunique().to_dict(),
        'q3_dog_frame_evidence_fraction':float(dog_records.status.isin(['detected','mask_tracked','flow_tracked']).mean()),
        'q3_longest_prediction_gap':int(dog_records.gap_frames.max()),
        'q3_temporal_identity_gates_passed':True,
        'q3_review_frames':[313,330,340,360],
        'q3_rejected_dog_proposals':len(q3_rejection_rows),
        'all_programmatic_checks_passed':True,
        'caveat':'Frame coverage and persistent IDs are not ground-truth tracking accuracy.'}
(OUT/'final_validation.json').write_text(json.dumps(checks,indent=2))
print('FINAL_QA_JSON',json.dumps(checks))
display(Markdown('### Track-lifetime audit (longest highway tracks)'))
display(track_audit.sort_values('observations',ascending=False).head(12))
display(Markdown('### Dog prediction intervals (observations and predictions are distinguished)'))
display(gap_table)

bundle=ROOT/'Project3_Q1_Q3_results.zip'
with ZipFile(bundle,'w',ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.suffix not in {'.gz','.npz'} and not path.name.endswith('_raw.mp4') and not path.name.startswith(('strong_detector_pilot','full_scale_detector_pilot','sam2_prompt_api_check','detection_pilot')):
            archive.write(path,path.relative_to(OUT))
print(f'Results archive: {bundle} ({bundle.stat().st_size/1024**2:.1f} MiB)')


FINAL_QA_JSON {"q1_feature_grids": 12, "q1_aggregate_gradcams": 4, "q1_channel_cam_grids": 4, "q2_processed_frames": 1501, "q2_fps": 25, "q3_processed_frames": 660, "q3_fps": 30, "q3_ids_per_class": {"dog": 1, "person": 1}, "q3_dog_frame_evidence_fraction": 0.9045454545454545, "q3_longest_prediction_gap": 63, "q3_temporal_identity_gates_passed": true, "q3_review_frames": [313, 330, 340, 360], "q3_rejected_dog_proposals": 94, "all_programmatic_checks_passed": true, "caveat": "Frame coverage and persistent IDs are not ground-truth tracking accuracy."}


### Track-lifetime audit (longest highway tracks)

    id  observations  first_frame  last_frame  max_detection_gap  \
21  33           385          760        1144                  0   
15  22           368          512         879                  0   
19  27           339          652         990                  0   
17  24           335          592         926                  0   
22  34           323          777        1099                  0   
9   13           320          107         426                  0   
28  46           320          923        1244                  2   
16  23           317          543         859                  0   
32  58           315         1087        1401                  0   
20  29           313          678         991                  1   
12  18           307          335         641                  0   
14  20           305          396         700                  0   

    max_center_motion_px_frame  
21                   46.736293  
15                   14.879852  
19              

### Dog prediction intervals (observations and predictions are distinguished)

        first_frame  last_frame  frames
status                                 
2               285         347      63

Results archive: /content/project3_q1_q3/Project3_Q1_Q3_results.zip (50.9 MiB)


The archive contains input/environment records, all figures, per-frame CSVs, validation summaries, and the two H.264 result videos. The notebook itself contains the executable implementation. The button below downloads a copy; it does not submit the assignment.

## References

- Project3_CV_2026_v5.pdf, Questions 1–3, and the supplied images/videos.
- [Torchvision models](https://docs.pytorch.org/vision/master/models.html): ResNet50 and DenseNet121, both using `IMAGENET1K_V1` weights.
- [Ultralytics tracking](https://docs.ultralytics.com/modes/track): YOLO11m-seg detection and ByteTrack association.
- [SAM 2 video segmentation](https://docs.ultralytics.com/models/sam-2): SAM2.1 base with automatic detector prompts.
- [OpenCV optical flow](https://docs.opencv.org/3.4.18/dc/d6b/group__video__track.html): Farneback dense flow and pyramidal Lucas–Kanade tracking.

The setup cell records the software versions, device, seed, input hashes, and video metadata used for this run.

## LLM Assistance Disclosure

OpenAI Codex assisted with planning, implementation, debugging, output review, and written analysis for Questions 1-3.